# C8 · Interpolation: a monsoon rainfall map from 109 stations

> **Time:** about 2 hours
> **You need:** Parts A and B, and Lessons C1 to C7, especially functions (A5), UTM 45N (B4), dissolve (B6), plotting with `extent=` (C2) and `rasterize` and `zonal_stats` (C7).
> **By the end you can:**
> 1. explain what interpolation is, and make **nearest-neighbour** (Voronoi) and **inverse distance weighting** (IDW) surfaces, with IDW written as your own function,
> 2. fit an **ordinary kriging** model with `pykrige`, and read its variogram and its map of uncertainty,
> 3. test every method honestly with **leave-one-out** validation and **RMSE**, never on the points used for fitting, and test whether elevation helps.

Rain gauges measure rainfall at a few points. Flood, landslide and water-supply studies need rainfall everywhere, in every cell of a grid. This lesson turns points into a surface in four ways, and then asks the most important question: *how good is each map?*

The 109 stations in `rain_gauges_monsoon_2024.csv` are real. They are the stations of the Department of Hydrology and Meteorology (DHM) that reported a total for the monsoon of 2024, June to September, in DHM's free seasonal summary (see `data/README.md`). It was a wet monsoon: across Nepal, 122% of the normal rainfall.

**Counting lines:** as in Part B, lines that only set up a plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** kriging fits its variogram with a numerical search, which can end in a slightly different place with other library versions. Your kriging results may then differ from the text by a few mm, or a few km of range.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 39 · [Course home](../README.md)  
← Previous: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) · Next: [C9 · Spatial autocorrelation: are neighbours alike?](C09_Spatial_autocorrelation.ipynb) →

In [ ]:
# Optional: marks your answers as you go. Delete this cell to turn it off.
%run ../tools/tutor_on.py

## 1. Why interpolate: from 109 stations to a grid

**Interpolation** estimates values at places where nothing was measured, from the places where something was. Here: the monsoon rainfall of 2024 in every cell of a grid over Nepal, from 109 stations.

Every method rests on one idea: places close together tend to be more alike than places far apart. Geographers call this the **first law of geography**.

Interpolation works with distances, so work in metres. Convert the stations to UTM 45N, as in B4. In degrees, a step east is shorter on the ground than a step north (B1), and distances would be distorted. Far-western Nepal lies in UTM zone 44, outside zone 45; distances there come out about half a percent too long, too little to matter here.

The grid to fill has **5 km** cells, built from scratch in the second cell:

- `dissolve()` merges the 77 districts into one outline of Nepal (B6).
- `np.arange` lists the x of every cell centre from west to east, and the y from north to south, the order of every raster (C1). `from_origin` makes the matching transform.
- `rasterize` (C7) burns the outline into the grid: 1 inside Nepal, 0 outside. `valid` is `True` inside.
- `xr.DataArray` wraps the grid with its coordinates, and `.rio.write_crs` adds the CRS, so the grid has the `.x`, `.y` and `.rio` tools of C3 to C7.

A coarse grid keeps the lesson fast: about 6,000 cells inside Nepal.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
import rioxarray  # noqa: F401  (adds the .rio methods)

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

rain = pd.read_csv(DATA / "rain_gauges_monsoon_2024.csv")     # DHM, June to September 2024
gauges_utm = gpd.GeoDataFrame(rain, geometry=gpd.points_from_xy(rain["lon"], rain["lat"]),
                              crs="EPSG:4326").to_crs(32645)
gauges_utm["x_m"] = gauges_utm.geometry.x
gauges_utm["y_m"] = gauges_utm.geometry.y
print(gauges_utm[["station", "elevation_m", "monsoon_mm", "x_m", "y_m"]].head(3).round(0))
print(gauges_utm["monsoon_mm"].describe().round(1))

In [ ]:
from rasterio import features

nepal = gpd.read_file(DATA / "nepal_districts.geojson").dissolve().to_crs(32645)   # one outline (B6)
xmin, ymin, xmax, ymax = nepal.total_bounds
cell_m = 5000
grid_x = np.arange(xmin + cell_m / 2, xmax, cell_m)      # cell centres, west to east
grid_y = np.arange(ymax - cell_m / 2, ymin, -cell_m)     # north to south, as in every raster (C1)
transform = rasterio.transform.from_origin(xmin, ymax, cell_m, cell_m)
inside = features.rasterize([(nepal.geometry.iloc[0], 1)], out_shape=(len(grid_y), len(grid_x)),
                            transform=transform, fill=0, dtype="uint8")       # 1 inside Nepal (C7)
valid = inside == 1                                      # True where the grid is inside Nepal
grid_5km = xr.DataArray(np.where(valid, 1.0, np.nan), coords={"y": grid_y, "x": grid_x},
                        dims=("y", "x")).rio.write_crs(32645)
print(grid_5km.shape, grid_5km.rio.resolution(), valid.sum(), "cells inside Nepal")

left, bottom, right, top = grid_5km.rio.bounds()
extent = [left, right, bottom, top]

fig, ax = plt.subplots(figsize=(11, 5))   # fig: the whole figure; ax: the map area inside it (A10)
nepal.boundary.plot(ax=ax, color="grey", linewidth=0.8)
dots = ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], c=gauges_utm["monsoon_mm"],
                  cmap="Blues", edgecolor="black", s=45, vmin=0, vmax=4000)
fig.colorbar(dots, ax=ax, label="Monsoon rainfall 2024 (mm)", shrink=0.7)
ax.set_title("109 DHM stations, June to September 2024")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The stations recorded between 284.1 mm (Jomsom) and 3,949.1 mm (Lumle) in the four monsoon months, 1,617.9 mm on average. The grid has 93 rows and 159 columns; 5,919 of its 14,787 cells lie inside Nepal.

The darkest dots crowd around Pokhara, in the middle of the country; the palest stand in the north-west and in the valleys behind the high mountains. Stations are dense in the Kathmandu Valley and along the main roads, and sparse in the remote north.

*Rainfall: Department of Hydrology and Meteorology, Government of Nepal, Preliminary precipitation and temperature summary, Monsoon (June–September) 2024.*

### Practice

**Q1 · one line:** How many cells does `grid_5km` hold in total? Use the attribute from A8 that counts all values.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
grid_5km.size
```

14,787 cells: 93 rows × 159 columns. 5,919 of them lie inside Nepal, and only 109 of those hold a station, about 1.8%. Interpolation has to estimate the other 98%.
</details>

**Q2 · two lines:** Which station is the wettest, and which the driest? Sort `gauges_utm` by `monsoon_mm` and print the first and the last row, with the `station`, `elevation_m` and `monsoon_mm` columns.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_rain = gauges_utm.sort_values("monsoon_mm")[["station", "elevation_m", "monsoon_mm"]]
print(by_rain.iloc[[0, -1]])
```

Jomsom is the driest (284.1 mm, at 2,741 m) and Lumle the wettest (3,949.1 mm, at 1,738 m). They are only about 55 km apart, on either side of the Annapurna range: Lumle faces the monsoon winds, Jomsom lies behind the mountains, in their **rain shadow**. `.iloc[[0, -1]]` takes the first and the last row in one go.
</details>

**Q3 · three lines:** How far is Jomsom from its nearest neighbour? Take Jomsom's point, measure the distance from every station to it with `.distance` (B8), leave out the zero (its distance to itself) and print the smallest, in km.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
jomsom = gauges_utm.loc[gauges_utm["station"] == "Jomsom", "geometry"].iloc[0]
dist_km = gauges_utm.distance(jomsom) / 1000
print(round(dist_km[dist_km > 0].min(), 1))
```

About 6.7 km, to Thakmarpha, with 305.1 mm: both lie in the dry upper Kali Gandaki valley. Distances in metres are correct here because `gauges_utm` is in UTM 45N.
</details>

## 2. Nearest neighbour: Voronoi polygons

The simplest method gives every place the value of its **nearest gauge**. The area that is closer to one gauge than to any other is that gauge's **Voronoi polygon**. In rainfall work they are also called **Thiessen polygons**, after the meteorologist who used them for exactly this.

- `gauges_utm.union_all()` joins the 109 points into one **MultiPoint** (B8).
- `shapely.voronoi_polygons(points, extend_to=box)` returns a collection of polygons that covers the box. `.geoms` lists them.
- `gpd.clip` (B7) cuts them to the outline of Nepal.
- The polygons come in no fixed order, so `gpd.sjoin` with `predicate="contains"` (B9) attaches each station's rainfall to the polygon that contains it.

Finally, `rasterize` (C7) burns the polygons into the 5 km grid, so that nearest neighbour becomes a raster like the other methods.

In [ ]:
import shapely
from shapely.geometry import box

study_area = box(left, bottom, right, top)             # the grid's rectangle (B2)
voronoi = shapely.voronoi_polygons(gauges_utm.union_all(), extend_to=study_area)
print(voronoi.geom_type, len(voronoi.geoms))

thiessen = gpd.GeoDataFrame(geometry=list(voronoi.geoms), crs=32645)
thiessen = gpd.clip(thiessen, nepal)                    # cut to Nepal's outline (B7)
thiessen = gpd.sjoin(thiessen, gauges_utm[["station", "monsoon_mm", "geometry"]], predicate="contains")
thiessen["area_km2"] = thiessen.area / 1000000
print(thiessen.sort_values("area_km2")[["station", "monsoon_mm", "area_km2"]].iloc[[0, -1]].round(1))

In [ ]:
pairs = list(zip(thiessen.geometry, thiessen["monsoon_mm"]))
nearest_mm = features.rasterize(pairs, out_shape=grid_5km.shape, transform=grid_5km.rio.transform(),
                                fill=np.nan, dtype="float32")
nearest_mm = np.where(valid, nearest_mm, np.nan)       # keep Nepal only (C5)

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(nearest_mm, extent=extent, cmap="Blues", vmin=0, vmax=4000)
thiessen.boundary.plot(ax=ax, color="grey", linewidth=0.4)
ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=8)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
fig.colorbar(im, ax=ax, label="Monsoon rainfall 2024 (mm)", shrink=0.8)
ax.set_title("Nearest neighbour: one value per Voronoi polygon")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The collection holds 109 polygons, one per station. After clipping, Khumaltar's polygon is the smallest (about 43.2 km², in the crowded Kathmandu Valley) and Dunai's the largest (about 7,448.5 km²): one station in Dolpa decides the rainfall of a large part of the dry north-west.

The map is a patchwork. Rainfall jumps by hundreds of millimetres at each polygon edge. Real rainfall does change fast in the mountains, but not along straight lines halfway between two stations. Still, Thiessen polygons are a classic way to turn gauge values into an average for a whole area, as Q2 shows.

`fill=np.nan` needs a float type, so the grid is `float32` (C7: the fill must fit the `dtype`).

### Practice

**Q1 · one line:** How many rows does `thiessen` have after the join? Print its length.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(thiessen))
```

109: every polygon contains exactly one station, so `sjoin` keeps all of them.
</details>

**Q2 · two lines:** A **Thiessen mean** weights each station by the area of its polygon. Work it out for Nepal, and print it next to the plain mean of the 109 stations.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
thiessen_mean_mm = (thiessen["monsoon_mm"] * thiessen["area_km2"]).sum() / thiessen["area_km2"].sum()
print(round(thiessen_mean_mm, 1), round(gauges_utm["monsoon_mm"].mean(), 1))
```

About 1,485.7 mm against a plain mean of 1,617.9 mm. Stations crowd into the hills and valleys where people live, many of them wet; the dry, empty north has few stations, each with a large polygon. Weighting by area gives those large polygons their share, so the Thiessen mean is lower.
</details>

**Q3 · three lines:** How much of Nepal gets more than 2,000 mm under nearest neighbour? Filter `thiessen`, add up the areas, and print the km² and the percentage of Nepal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
wet = thiessen[thiessen["monsoon_mm"] > 2000]
wet_km2 = wet["area_km2"].sum()
print(round(wet_km2, 1), round(wet_km2 / thiessen["area_km2"].sum() * 100, 1))
```

About 24,763.2 km², or 16.7% of Nepal. With polygons, "how much area" questions are sums of polygon areas, as in Part B.
</details>

## 3. Inverse distance weighting, written as a function

Nearest neighbour listens to one gauge only. **Inverse distance weighting** (IDW) listens to all of them, the near ones loudest.

Each gauge gets a **weight** w = 1 / dᵖ, where d is its distance from the place you estimate and p is the **power**. The estimate is the **weighted mean**: the sum of w × value, divided by the sum of w. Halve the distance, and with p = 2 the weight becomes four times larger.

At a gauge itself d = 0, and 1 / 0 has no answer. The function below returns that gauge's own value instead. So IDW **honours** the measurements: it is an **exact interpolator**.

You write IDW as a function (A5) for one place, with NumPy arrays (A8) holding all 109 stations at once. A second function calls it for every cell of the grid (functions calling functions, A5).

**Predict:** what will `idw` return at the exact position of the first station, Baglung, which recorded 2,060.6 mm?

In [ ]:
xs_m = gauges_utm["x_m"].to_numpy()
ys_m = gauges_utm["y_m"].to_numpy()
rain_mm = gauges_utm["monsoon_mm"].to_numpy().astype(float)

def idw(x_m, y_m, xs_m, ys_m, values, power=2):
    """Inverse distance weighted estimate at one place (x_m, y_m)."""
    dist_m = np.sqrt((xs_m - x_m) ** 2 + (ys_m - y_m) ** 2)   # distance to every gauge
    if dist_m.min() == 0:                  # exactly on a gauge: return its value
        return values[dist_m.argmin()]
    weights = 1 / dist_m ** power
    return (weights * values).sum() / weights.sum()

# Kathmandu Durbar Square (approximate), converted to UTM 45N as in B4
durbar_utm = gpd.GeoSeries(gpd.points_from_xy([85.307], [27.704]), crs=4326).to_crs(32645)
durbar_x_m, durbar_y_m = durbar_utm.x.iloc[0], durbar_utm.y.iloc[0]

print(gauges_utm["station"].iloc[0], idw(xs_m[0], ys_m[0], xs_m, ys_m, rain_mm))   # on the first gauge
print(round(idw(durbar_x_m, durbar_y_m, xs_m, ys_m, rain_mm), 1))                  # Durbar Square

In [ ]:
def idw_surface(xs_m, ys_m, values, power=2):
    """IDW estimate for every cell of grid_5km inside Nepal; NaN elsewhere."""
    grid_x = grid_5km.x.values               # x of each column's centre
    grid_y = grid_5km.y.values               # y of each row's centre, north to south
    surface = np.full(grid_5km.shape, np.nan) # a grid of NaN to fill in
    for row in range(len(grid_y)):
        for col in range(len(grid_x)):
            if valid[row, col]:
                surface[row, col] = idw(grid_x[col], grid_y[row], xs_m, ys_m, values, power)
    return surface

idw2_mm = idw_surface(xs_m, ys_m, rain_mm)
print(round(np.nanmin(idw2_mm), 1), round(np.nanmax(idw2_mm), 1), round(np.nanmean(idw2_mm), 1))

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(idw2_mm, extent=extent, cmap="Blues", vmin=0, vmax=4000)
ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=8)
fig.colorbar(im, ax=ax, label="Monsoon rainfall 2024 (mm)", shrink=0.8)
ax.set_title("Inverse distance weighting, power 2")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

On Baglung the function returns exactly 2,060.6 mm, the station's own value. At Durbar Square, between the stations of the Kathmandu Valley, IDW estimates about 1,729.1 mm.

`np.full(shape, np.nan)` makes a grid of the given shape filled with `NaN`; the loops then fill the cells inside Nepal, row by row, as in A3. The surface runs from about 290.2 to 3,804.2 mm, always inside the range of the stations: a weighted mean can never be larger than the largest value, or smaller than the smallest.

The map is smooth, but many stations sit in a small ring of their own colour, a **bull's-eye**, such as the dark ring around Lumle and the pale ones in the north. That is the typical look of IDW.

### Practice

**Q1 · one line:** Estimate the rainfall at Durbar Square with `power=1` instead of 2.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
idw(durbar_x_m, durbar_y_m, xs_m, ys_m, rain_mm, power=1)
```

About 1,738.5 mm, a little more than with power 2. With a lower power, far stations count for more; here the farther stations happen to be slightly wetter than the valley's own.
</details>

**Q2 · two lines:** Read the IDW estimate at a station from the **grid**, not from the function: find Lumle's row in the arrays with `gauges_utm.index[gauges_utm["station"] == "Lumle"][0]`, its row and column in the grid with `rasterio.transform.rowcol` (C1) and `grid_5km.rio.transform()`, then print `idw2_mm` at that cell next to Lumle's measured value. (Three lines are fine here.)

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
i = gauges_utm.index[gauges_utm["station"] == "Lumle"][0]
row, col = rasterio.transform.rowcol(grid_5km.rio.transform(), xs_m[i], ys_m[i])
print(round(idw2_mm[row, col], 1), rain_mm[i])
```

3,804.2 mm in the grid and 3,949.1 mm measured. The cell's centre is up to 3.5 km from Lumle, and at that distance the drier stations around it already pull the estimate down. IDW honours a station exactly only at the station itself.
</details>

**Q3 · three lines:** Where do IDW and nearest neighbour disagree most? Make the absolute difference of `idw2_mm` and `nearest_mm`, find its largest value with `np.nanmax`, and its row and column with `np.nanargmax` and `np.unravel_index` (A8).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
diff_mm = np.abs(idw2_mm - nearest_mm)
row, col = np.unravel_index(np.nanargmax(diff_mm), diff_mm.shape)
print(round(np.nanmax(diff_mm), 1), row, col)
```

About 2,157.7 mm, at row 44, column 73, near 83.8° E, 28.5° N, on the Annapurna massif. The cell lies in Lumle's Voronoi polygon, 24 km from Lumle, so nearest neighbour copies Lumle's 3,949 mm there. IDW blends Lumle with the dry stations of the Kali Gandaki valley, which are only a little farther away.
</details>

## 4. Choosing the power

The power p decides how fast a gauge's influence fades with distance.

- **p = 1**: far gauges still count a lot. The surface is smooth and pulled towards the mean of all gauges.
- **p = 2**: the most common choice, and the default of many GIS programs.
- **Large p**: the nearest gauge takes almost all the weight, and the map turns into the nearest-neighbour patchwork.

All three are "correct" IDW maps. Looking at them cannot tell you which is best; step 5 measures it.

**Predict:** of the three maps below, which one will look most like the Voronoi map of step 2?

In [ ]:
idw1_mm = idw_surface(xs_m, ys_m, rain_mm, power=1)
idw4_mm = idw_surface(xs_m, ys_m, rain_mm, power=4)

fig, axes = plt.subplots(3, 1, figsize=(9, 11))
for ax, surface, power in zip(axes, [idw1_mm, idw2_mm, idw4_mm], [1, 2, 4]):
    im = ax.imshow(surface, extent=extent, cmap="Blues", vmin=0, vmax=4000)
    ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=4)
    ax.set_title(f"IDW, power {power}")
    ax.set_xticks([])
    ax.set_yticks([])
    print(power, "standard deviation of the surface:", round(np.nanstd(surface), 1), "mm")
fig.colorbar(im, ax=axes, label="Monsoon rainfall 2024 (mm)", shrink=0.6)
plt.show()

With power 1 the map is soft and mostly middle blue: the **standard deviation** of its cells (how much they spread around their mean, A9) is only about 161.7 mm. With power 2 it is about 372.8 mm, and with power 4 about 560.6 mm, with sharp patches around each station: the most Voronoi-like of the three.

The same colour limits (`vmin`, `vmax`) on all three maps make them comparable, so one colour bar serves all. `ax.set_xticks([])` hides the axis numbers to save space; the grid is the same 5 km UTM grid as before.

### Practice

**Q1 · one line:** With power 2, how much more weight does a gauge 2 km away get than a gauge 4 km away? Work it out with the formula w = 1 / dᵖ.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(1 / 2 ** 2) / (1 / 4 ** 2)
```

4.0: half the distance, four times the weight. Units cancel in the ratio, so km work as well as metres here.
</details>

**Q2 · two lines:** Work out the same ratio, 2 km against 4 km, for power 1 and for power 4.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print((1 / 2 ** 1) / (1 / 4 ** 1))
print((1 / 2 ** 4) / (1 / 4 ** 4))
```

2.0 for power 1 and 16.0 for power 4. Each step up in power doubles the advantage of the nearer gauge again, which is why large powers end in patches.
</details>

**Q3 · three lines:** Make an IDW surface with power 8 and count the cells inside Nepal where it lies within 50 mm of `nearest_mm`. Print the count and the percentage of the 5,919 cells inside Nepal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
idw8_mm = idw_surface(xs_m, ys_m, rain_mm, power=8)
close = np.abs(idw8_mm - nearest_mm)[valid] <= 50
print(close.sum(), round(close.mean() * 100, 1))
```

4,094 cells, about 69.2% of the 5,919 inside Nepal, are within 50 mm. With power 8, IDW is close to nearest neighbour almost everywhere except near the polygon edges, where it still blends.
</details>

## 5. Testing honestly: leave-one-out and RMSE

An exact interpolator returns each gauge's own value, so checking it **at** the gauges proves nothing. An honest test predicts gauges the method has **not** seen.

**Leave-one-out cross-validation** does this with every gauge in turn: hide station 1, estimate it from the other 108, note the **error** (estimate − measured value); then hide station 2, and so on, 109 times.

The **RMSE** (root mean square error) sums up the 109 errors: square them, take the mean, then the square root. It is a typical size of error, in mm, with big misses counting extra. The **mean error** shows **bias**: whether a method guesses too high or too low on average.

Always compare with a **baseline**, a method that ignores location: here, the mean of the other 108 stations. A map is only worth drawing if it beats the baseline. `keep = np.arange(n) != i` is a mask (A8) that is `True` for every station except number `i`.

In [ ]:
def rmse(errors):
    """Root mean square error of an array of errors."""
    return np.sqrt(np.mean(errors ** 2))

def loo_errors(power):
    """Leave-one-out errors (mm) of IDW with the given power."""
    errors_mm = []
    for i in range(len(rain_mm)):
        keep = np.arange(len(rain_mm)) != i          # every gauge except number i
        estimate = idw(xs_m[i], ys_m[i], xs_m[keep], ys_m[keep], rain_mm[keep], power)
        errors_mm.append(estimate - rain_mm[i])
    return np.array(errors_mm)

n = len(rain_mm)
baseline_mm = np.array([rain_mm[np.arange(n) != i].mean() - rain_mm[i] for i in range(n)])
print(f"baseline (mean of the others): RMSE {rmse(baseline_mm):.1f} mm")
for power in [1, 2, 3, 4, 8]:
    errors_mm = loo_errors(power)
    print(f"IDW power {power}: RMSE {rmse(errors_mm):.1f} mm, mean error {errors_mm.mean():.1f} mm")

The baseline misses by about 632.6 mm. IDW does best with power 3 (about 425.7 mm), then power 4 (429.7 mm), power 2 (447.0 mm) and power 8 (450.7 mm); power 1 is clearly worse (543.2 mm).

The mean errors are small, between −18 and +10 mm: IDW has hardly any bias here. The best IDW map beats "use the average everywhere" by about a third. That is useful, but a typical error of over 400 mm, a quarter of the average rainfall, shows how much the monsoon varies between neighbouring valleys.

Be modest with such numbers: powers 2 to 8 are within 25 mm of each other, and moving a few stations could reorder them.

### Practice

**Q1 · one line:** Which station does IDW with power 2 miss by the most? Use `np.abs(...).argmax()` on `loo_errors(2)` to pick the row of `gauges_utm["station"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gauges_utm["station"].iloc[np.abs(loo_errors(2)).argmax()]
```

Lumle, the wettest station. Hidden, it can only be estimated from its neighbours, which are all drier, even Pokhara Airport (3,246.6 mm, 19 km away), so any average of them falls short.
</details>

**Q2 · two lines:** Store the power-2 leave-one-out errors in a new column `loo_err_mm` of `gauges_utm`, then print the mean error of the stations above 2,000 m.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gauges_utm["loo_err_mm"] = loo_errors(2)
print(round(gauges_utm.loc[gauges_utm["elevation_m"] > 2000, "loo_err_mm"].mean(), 1))
```

About +186.6 mm: IDW **overestimates** the high stations. Most of them stand in dry valleys behind the mountains, such as Jomsom, Dunai and Jumla, and their wetter neighbours to the south pull the estimates up.
</details>

**Q3 · three lines:** Draw a scatter plot (A10) of `elevation_m` (x) against `loo_err_mm` (y), with a horizontal line at 0 (`ax.axhline(0)`) and labelled axes.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(gauges_utm["elevation_m"], gauges_utm["loo_err_mm"])
ax.axhline(0, color="grey")
ax.set_xlabel("Station elevation (m)")
ax.set_ylabel("Leave-one-out error (mm)")
plt.show()
```

Most stations below about 2,000 m scatter on both sides of zero, while most high stations lie above it: overestimated, as Q2 found. The wettest stations, such as Lumle, sit far below zero. Errors with a pattern are a hint that something is missing; step 7 tests whether elevation is that something.
</details>

## 6. Kriging, tested on gauges it has not seen

**Kriging** is also a weighted mean, but it learns its weights from the data. First it measures how different two gauges are, on average, as the distance between them grows. Half the mean squared difference is the **semivariance**; plotted against distance it is the **variogram**. A curve fitted to it has three numbers:

- **nugget**: the difference left even between gauges very close together (noise, very local effects), in mm²,
- **range**: the distance beyond which gauges are no more alike than any two random gauges,
- **partial sill**: how far the curve rises from the nugget to its plateau, in mm².

`OrdinaryKriging(xs, ys, values, variogram_model="spherical", nlags=20)` from `pykrige` fits the curve. `nlags=20` sorts the pairs of stations into 20 distance classes. The default is 6, and over 700 km of Nepal each class would then be about 120 km wide: too coarse to see how fast rainfall changes from valley to valley. `.execute("grid", grid_x, grid_y)` fills a grid; `.execute("points", xs, ys)` estimates at points. Both also return the **kriging variance**: an uncertainty for every estimate.

The honest rule from step 5 still holds: test only on gauges the model has **not** seen. A common mistake is to fit kriging on all points and then test it on some of those same points. The third cell makes that mistake on purpose.

In [ ]:
from pykrige.ok import OrdinaryKriging

ok = OrdinaryKriging(xs_m, ys_m, rain_mm, variogram_model="spherical", nlags=20)
psill, range_m, nugget = ok.variogram_model_parameters
print(f"partial sill {psill:.0f} mm², range {range_m / 1000:.1f} km, nugget {nugget:.0f} mm²")

dist_m = np.linspace(0, ok.lags.max(), 100)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ok.lags / 1000, ok.semivariance, "o", label="From the gauges")
ax.plot(dist_m / 1000, ok.variogram_function(ok.variogram_model_parameters, dist_m), label="Fitted spherical curve")
ax.set_xlabel("Distance between gauges (km)")
ax.set_ylabel("Semivariance (mm²)")
ax.legend()
plt.show()

In [ ]:
krig_mm, krig_var = ok.execute("grid", grid_5km.x.values, grid_5km.y.values)
krig_mm = np.where(valid, krig_mm, np.nan)
krig_sd_mm = np.where(valid, np.sqrt(krig_var), np.nan)     # standard deviation, in mm
print(krig_mm.shape, round(np.nanmin(krig_mm), 1), round(np.nanmax(krig_mm), 1))
print(round(np.nanmin(krig_sd_mm), 1), round(np.nanmax(krig_sd_mm), 1))

fig, axes = plt.subplots(2, 1, figsize=(10, 9))
im0 = axes[0].imshow(krig_mm, extent=extent, cmap="Blues", vmin=0, vmax=4000)
im1 = axes[1].imshow(krig_sd_mm, extent=extent, cmap="Oranges")
for ax in axes:
    ax.scatter(gauges_utm["x_m"], gauges_utm["y_m"], color="black", s=6)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
fig.colorbar(im0, ax=axes[0], label="Monsoon rainfall 2024 (mm)", shrink=0.8)
fig.colorbar(im1, ax=axes[1], label="Kriging standard deviation (mm)", shrink=0.8)
axes[0].set_title("Ordinary kriging estimate")
axes[1].set_title("Its uncertainty")
plt.show()

In [ ]:
krig_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i                 # train on the other gauges only
    ok_rest = OrdinaryKriging(xs_m[keep], ys_m[keep], rain_mm[keep], variogram_model="spherical", nlags=20)
    estimate, variance = ok_rest.execute("points", xs_m[i:i + 1], ys_m[i:i + 1])
    krig_errors_mm.append(estimate[0] - rain_mm[i])
krig_errors_mm = np.array(krig_errors_mm)
print("kriging, leave-one-out RMSE:", round(rmse(krig_errors_mm), 1), "mm")

# THE MISTAKE, ON PURPOSE: "testing" on the gauges the model was fitted on
same_mm, same_var = ok.execute("points", xs_m, ys_m)
print("kriging, tested on its own gauges:", round(rmse(same_mm - rain_mm), 1), "mm")

The fitted curve has a nugget of about 10,584 mm², a partial sill of about 337,952 mm² and a range of about 82.9 km. Stations closer than about 83 km are more alike than stations farther apart. The dots scatter a lot: monsoon rainfall can differ greatly between two valleys a few kilometres apart, so even close pairs differ.

The grid comes back with one row per y value, in the order you gave them. `grid_5km.y` runs north to south, so the result is north-up (C1). The estimate ranges from about 271.6 to 3,723.6 mm. The wettest station measured 3,949 mm, but no cell reaches it: with a nugget, kriging does not force a sharp bull's-eye at each station. The standard deviation is about 141.4 mm next to the stations and up to about 599.3 mm far from them, in the remote north-west. **That second map is kriging's real gift: it shows where the first map can be trusted.**

Refitted 109 times on 108 stations, kriging scores an RMSE of about 458.8 mm: a little **worse** than IDW with power 2 (447.0 mm). Kriging is not automatically better: its smooth variogram cannot see the mountain ranges that split wet slopes from dry valleys either. "Tested" on its own stations it scores 0.0 mm, a perfect and meaningless score, because kriging is an exact interpolator too. Any error measured on the fitting points says nothing about the map.

### Practice

**Q1 · one line:** The nugget is in mm². Print its square root, in mm, and the range in km, both rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(np.sqrt(nugget), 1), round(range_m / 1000, 1))
```

About 102.9 mm and 82.9 km. The square root of the nugget is the size of the purely local variation in this model: differences between nearby stations that no interpolation can predict.
</details>

**Q2 · two lines:** Estimate the rainfall at Durbar Square with `ok.execute("points", ...)`, using `[durbar_x_m]` and `[durbar_y_m]` (lists of one value). Print the estimate and its standard deviation.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
durbar_mm, durbar_var = ok.execute("points", [durbar_x_m], [durbar_y_m])
print(round(float(durbar_mm[0]), 1), round(float(np.sqrt(durbar_var[0])), 1))
```

About 1,771.7 mm, give or take about 219.8 mm (one standard deviation). IDW said about 1,729.1 mm, and the valley's own stations measured 1,817.0 mm (Kathmandu Airport) and 1,655.0 mm (Khumaltar). The standard deviation is among the lowest on the map: several stations stand close to Durbar Square, and kriging says so.
</details>

**Q3 · three lines:** Fit the same leave-one-out test with `variogram_model="linear"` (and `nlags=20`), a straight-line variogram, and print its RMSE. Loop as in the third example cell.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
lin_errors = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    ok_lin = OrdinaryKriging(xs_m[keep], ys_m[keep], rain_mm[keep], variogram_model="linear", nlags=20)
    lin_errors.append(ok_lin.execute("points", xs_m[i:i + 1], ys_m[i:i + 1])[0][0] - rain_mm[i])
print(round(rmse(np.array(lin_errors)), 1))
```

About 558.5 mm, much worse than the spherical model (458.8 mm). The choice of variogram model matters, and leave-one-out tells you which one to trust. `.execute(...)[0][0]` takes the estimates, then the first one.
</details>

## 7. Does elevation help?

Distance-only methods beat the baseline by about a third. Can a **helper variable** do better? In one valley, such as Kathmandu's, rainfall often rises with height, because air cools and drops its water as it climbs the slopes. Across all of Nepal the picture is different: the monsoon arrives from the south-east, the first high ridges catch most of its rain, and the valleys behind them, such as Mustang, lie in a **rain shadow**.

The stations carry their elevation. `np.polyfit(x, y, 1)` fits the best straight line, y = a × x + b, and returns the slope `a` and the intercept `b`. `np.corrcoef(x, y)[0, 1]` gives the correlation, from −1 to +1 (C9 uses it again).

The test is the same as before: leave one station out, fit the line on the other 108, and predict the hidden one.

In [ ]:
elev_m = gauges_utm["elevation_m"].to_numpy().astype(float)
a_mm_per_m, b_mm = np.polyfit(elev_m, rain_mm, 1)
print(f"rainfall = {a_mm_per_m:.3f} x elevation + {b_mm:.1f}")
print("correlation:", round(np.corrcoef(elev_m, rain_mm)[0, 1], 2))

line_m = np.array([elev_m.min(), elev_m.max()])
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(elev_m, rain_mm)
ax.plot(line_m, a_mm_per_m * line_m + b_mm, color="crimson", label="Fitted line")
for name in ["Lumle", "Pokhara Airport", "Jomsom", "Thakmarpha"]:
    row = gauges_utm[gauges_utm["station"] == name].iloc[0]
    ax.annotate(name, (row["elevation_m"], row["monsoon_mm"]), xytext=(5, 0), textcoords="offset points", fontsize=8)
ax.set_xlabel("Station elevation (m)")
ax.set_ylabel("Monsoon rainfall 2024 (mm)")
ax.legend()
plt.show()

In [ ]:
trend_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    a, b = np.polyfit(elev_m[keep], rain_mm[keep], 1)      # fitted without gauge i
    trend_errors_mm.append(a * elev_m[i] + b - rain_mm[i])
print("elevation line, leave-one-out RMSE:", round(rmse(np.array(trend_errors_mm)), 1), "mm")

In [ ]:
annapurna = gauges_utm[gauges_utm["station"].isin(["Lumle", "Pokhara Airport", "Jomsom", "Thakmarpha"])]
print(annapurna[["station", "district", "elevation_m", "monsoon_mm"]])
print(round(annapurna.distance(annapurna.geometry.iloc[0]).max() / 1000, 1), "km between the farthest two")

The line **falls**: about 0.218 mm less rain per metre of height, and the correlation is weak, −0.28. Its leave-one-out RMSE is about 612.9 mm, hardly better than the baseline (632.6 mm), and far worse than IDW (447.0 mm). Across Nepal, elevation alone is a poor helper.

The four Annapurna stations show why. Pokhara Airport (827 m, 3,246.6 mm) and Lumle (1,738 m, 3,949.1 mm) face the monsoon; Jomsom (2,741 m, 284.1 mm) and Thakmarpha (2,655 m, 305.1 mm) lie behind the range. Height matters, but so does which side of the mountains a place is on, and a straight line through all of Nepal cannot know that. With a national DEM, a model could also use slope direction or the distance to the main ridge; the lesson here is to **test** every helper before you trust it.

### Practice

**Q1 · one line:** What does the fitted line give for 1,400 m, the approximate elevation of Kathmandu on the facts sheet?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
a_mm_per_m * 1400 + b_mm
```

About 1,538.0 mm. Kathmandu's valley stations measured 1,655.0 and 1,817.0 mm: a line fitted across the whole country knows nothing about one valley.
</details>

**Q2 · two lines:** How many stations lie above 2,000 m, and how does their mean monsoon rainfall compare with the stations below? Make a mask with `>`, and print the count and both means; `~mask` gives the stations below (A9).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
high = gauges_utm["elevation_m"] > 2000
print(high.sum(), round(gauges_utm.loc[high, "monsoon_mm"].mean(), 1), round(gauges_utm.loc[~high, "monsoon_mm"].mean(), 1))
```

16 stations, with 1,111.8 mm on average, against 1,705.0 mm for the stations below 2,000 m. In Nepal the high stations are drier, because most of them lie north of the main ridges, out of reach of the monsoon winds.
</details>

**Q3 · three lines:** The **residual** of a station is its measured rainfall minus the line's value at its elevation. Compute the 109 residuals with the line fitted to all stations, and print the largest one in absolute value and the station it belongs to.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
residual_mm = rain_mm - (a_mm_per_m * elev_m + b_mm)
biggest = np.abs(residual_mm).argmax()
print(gauges_utm["station"].iloc[biggest], round(residual_mm[biggest], 1))
```

Lumle has the largest residual, about +2,484.7 mm: far wetter than its elevation suggests. It is also the station IDW missed most in step 5. Residuals are what elevation does not explain; Review 2 tries to interpolate them.
</details>

## 8. When it breaks: `math` functions on arrays

In A5 you used `math.sqrt` for single numbers. Here it is used for the distances from one place to all 109 stations at once. Run the cell.

In [ ]:
import math

dist_m = math.sqrt((xs_m - durbar_x_m) ** 2 + (ys_m - durbar_y_m) ** 2)

Read the last line: `TypeError: only 0-dimensional arrays can be converted to Python scalars`.

- A **scalar** is a single number. A **0-dimensional array** is NumPy's name for an array that holds only one number.
- `math.sqrt` takes one number only. It tried to turn the array of 109 squared distances into one number, and could not.

The `math` module from A5 works on single numbers; NumPy functions work on whole arrays at once (A8). The fix is `np.sqrt`, exactly as in the `idw` function. The same goes for `math.exp`, `math.log` and the others: use `np.exp`, `np.log` with arrays.

**Try it:** change `math.sqrt` to `np.sqrt` and run the cell again.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A hold-out test

Leave-one-out is one honest test; a **hold-out** test is another. Put every fourth station aside as test stations with `gauges_utm.iloc[::4]` and use the rest, `gauges_utm.drop(test.index)`, for training. Fit on the training stations only, then print the RMSE on the test stations of three methods: the training mean, IDW with power 2, and the elevation line.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
test = gauges_utm.iloc[::4]
train = gauges_utm.drop(test.index)
print(len(test), "test gauges,", len(train), "training gauges")
tx, ty, tv = train["x_m"].to_numpy(), train["y_m"].to_numpy(), train["monsoon_mm"].to_numpy().astype(float)
obs_mm = test["monsoon_mm"].to_numpy()

print("mean:", round(rmse(tv.mean() - obs_mm), 1))
idw_test = np.array([idw(x, y, tx, ty, tv) for x, y in zip(test["x_m"], test["y_m"])])
print("IDW: ", round(rmse(idw_test - obs_mm), 1))
a, b = np.polyfit(train["elevation_m"], tv, 1)
print("line:", round(rmse(a * test["elevation_m"].to_numpy() + b - obs_mm), 1))
```

There are 28 test and 81 training stations. The training mean misses the test stations by about 464.6 mm, IDW by about 287.6 mm and the elevation line by about 426.7 mm. IDW wins, as in leave-one-out, but every number is different: a hold-out score depends a lot on which stations were set aside. Leave-one-out uses every station once, which is why it suits small data sets.
</details>

### Review 2 · Line plus interpolated residuals

Combine steps 3 and 7: for each station left out, fit the elevation line on the other 108, interpolate their **residuals** with IDW (power 2) to the hidden station, and add the interpolated residual to the line's value. Print the leave-one-out RMSE and compare it with the line alone (612.9 mm) and IDW alone (447.0 mm).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
combo_errors_mm = []
for i in range(len(rain_mm)):
    keep = np.arange(len(rain_mm)) != i
    a, b = np.polyfit(elev_m[keep], rain_mm[keep], 1)
    residuals = rain_mm[keep] - (a * elev_m[keep] + b)
    residual_here = idw(xs_m[i], ys_m[i], xs_m[keep], ys_m[keep], residuals)
    combo_errors_mm.append(a * elev_m[i] + b + residual_here - rain_mm[i])
print(round(rmse(np.array(combo_errors_mm)), 1))
```

About 450.4 mm: far better than the line alone (612.9 mm), but no better than IDW alone (447.0 mm). The line adds almost nothing, so this combination is IDW with extra steps. In a single valley, where rain does rise with height, the same combination can help; when kriging interpolates the residuals, the method is called **regression kriging**. The lesson: test every extra step; do not assume it helps.
</details>

### Review 3 · Monsoon rainfall per province

Use `zonal_stats` (C7) to find the mean of the IDW map `idw2_mm` in each province. Read `nepal_provinces.geojson`, convert it to UTM 45N, and add the province names from `province_names.csv` (B6). Print a table of province and mean rainfall, wettest first.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
from rasterstats import zonal_stats

provinces = gpd.read_file(DATA / "nepal_provinces.geojson").to_crs(32645)
names = pd.read_csv(DATA / "province_names.csv")
stats = zonal_stats(provinces, idw2_mm, affine=grid_5km.rio.transform(), nodata=np.nan, stats=["mean", "count"])
table = pd.DataFrame(stats)
table["province_code"] = provinces["province_code"]
table = table.merge(names[["province_code", "province"]], on="province_code")
print(table.sort_values("mean", ascending=False).round(1))
```

Bagmati comes out wettest (about 1,838.3 mm), just ahead of Gandaki (1,817.7 mm); Karnali is by far the driest (1,154.1 mm), with its large dry north. Look at the `count` column too: Madhesh's mean rests on 378 cells of 5 km, and on the few stations near them. Report the count with every zonal mean, and remember that every value is an IDW estimate, not a measurement.
</details>

### Review 4 · Save the IDW map and read it back

Write `idw2_mm` to `../outputs/c08_monsoon_idw.tif` as in C1: one band, `float32`, the CRS and transform of `grid_5km`, and `nodata=np.nan`. Read it back and sample it at Durbar Square with `src.sample`. Use the UTM coordinates `durbar_x_m`, `durbar_y_m`, because the file is in UTM 45N (C7).

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
profile = {"driver": "GTiff", "height": idw2_mm.shape[0], "width": idw2_mm.shape[1], "count": 1,
           "dtype": "float32", "crs": grid_5km.rio.crs, "transform": grid_5km.rio.transform(), "nodata": np.nan}
with rasterio.open(OUT / "c08_monsoon_idw.tif", "w", **profile) as dst:
    dst.write(idw2_mm.astype("float32"), 1)

with rasterio.open(OUT / "c08_monsoon_idw.tif") as src:
    print(src.crs, src.res, src.transform.e < 0)
    print(round(float(list(src.sample([(durbar_x_m, durbar_y_m)]))[0][0]), 1), "mm at Durbar Square")
```

The file is in EPSG:32645 with 5 km cells, and `transform.e < 0` is `True`: north is up (C1). Durbar Square's 5 km cell reads about 1,696.3 mm, not the 1,729.1 mm that `idw` gave in step 3: the file holds the estimate for the cell's centre, up to 3.5 km away, not for the square itself.
</details>

### Review 5 · Where would a 110th station help most?

A new station helps most where the kriging map is least certain. Find the cell with the largest `krig_sd_mm` (`np.nanargmax` and `np.unravel_index`, A8), take its x and y from `grid_5km.x.values` and `grid_5km.y.values`, convert that point to longitude and latitude (B4), and print them with the standard deviation.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = np.unravel_index(np.nanargmax(krig_sd_mm), krig_sd_mm.shape)
x_m, y_m = grid_5km.x.values[col], grid_5km.y.values[row]
spot = gpd.GeoSeries(gpd.points_from_xy([x_m], [y_m]), crs=32645).to_crs(4326)
print(row, col, round(krig_sd_mm[row, col], 1), "mm")
print(round(spot.x.iloc[0], 3), round(spot.y.iloc[0], 3))
```

The most uncertain cell is at row 17, column 56, near 82.92° E, 29.71° N, in the remote north of Dolpa, with a standard deviation of about 599.3 mm. The north-west has few stations, and they are far apart. In practice you would also weigh access, safety and what the study needs, but the kriging variance is a sound place to start.
</details>

## Recap

- **Interpolation** estimates values between measured points. Work in a projected CRS (UTM 45N), because every method uses distances.
- **Nearest neighbour** gives Voronoi (Thiessen) patches; **IDW** is a weighted mean with weights 1 / dᵖ, smoother with a small power and patchier with a large one. Both are exact at the gauges.
- **Kriging** learns its weights from a **variogram** (nugget, range, partial sill) and also gives a **variance**: a map of where its estimate can be trusted.
- Test honestly: **leave-one-out** or **hold-out**, with **RMSE** and a **baseline**. Never test on the points used for fitting; that score is meaningless.
- A helper variable can beat every distance method, but only where it really explains the values: across Nepal, elevation alone does not. Test every helper, and remember that with 109 stations in the mountains, every map is rough.

### Words you learned

| Word | Meaning |
| --- | --- |
| interpolation | Estimating values at unmeasured places from measured ones |
| first law of geography | Near things are more alike than distant things |
| Voronoi / Thiessen polygon | The area closer to one point than to any other point |
| nearest neighbour | Interpolation that copies the value of the closest point |
| inverse distance weighting (IDW) | A weighted mean with weights 1 / dᵖ: nearer points count more |
| power (IDW) | The exponent p that sets how fast a point's weight falls with distance |
| weighted mean | A mean in which some values count more than others |
| exact interpolator | A method that returns the measured value at each measured point |
| bull's-eye | A ring of colour around a single point on an interpolated map |
| leave-one-out cross-validation | Hiding each point in turn and predicting it from the others |
| hold-out test | Setting some points aside for testing and fitting on the rest |
| RMSE | Root mean square error: the square root of the mean squared error |
| bias | A tendency to estimate too high or too low on average |
| baseline | A simple method that a better method must beat |
| kriging | Interpolation whose weights come from a fitted variogram |
| semivariance / variogram | Half the mean squared difference between points / its curve against distance |
| nugget, range, partial sill | The variogram's value near zero distance, the distance where it levels off, and its rise |
| kriging variance | Kriging's estimate of its own uncertainty in each cell |
| trend surface | A surface computed from a helper variable, here a line fitted to elevation |
| residual | A measured value minus the value a model predicts |
| extrapolation | Estimating beyond the range of the data |
| scalar | A single number |

## Next

The last lesson of Part C is **C9 · Spatial autocorrelation**: how to measure that nearby places are alike, the idea behind every method in this lesson, and why it matters for testing models on maps.

<!-- pygile-nav -->
**PyGILE Course** · lesson 32 of 39 · [Course home](../README.md)  
← Previous: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) · Next: [C9 · Spatial autocorrelation: are neighbours alike?](C09_Spatial_autocorrelation.ipynb) →